In [25]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

In [26]:
[x for x in timm.list_models("*xception*")]

['legacy_xception',
 'xception41',
 'xception41p',
 'xception65',
 'xception65p',
 'xception71']

In [27]:
backbone = timm.create_model(
    "legacy_xception",
    pretrained=True,
    features_only=True,
    out_indices=(1,2,3,4),
)
backbone.eval()

FeatureHookNet(
  (body): Xception(
    (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act2): ReLU(inplace=True)
    (block1): Block(
      (skip): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (skipbn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (rep): Sequential(
        (0): SeparableConv2d(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
          (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        )
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inpla

In [28]:
x = torch.randn(2,3,256,256)

with torch.no_grad():
  feats = backbone(x)

for i,f in enumerate(feats):
  print(i,f.shape)

0 torch.Size([2, 128, 63, 63])
1 torch.Size([2, 256, 32, 32])
2 torch.Size([2, 728, 16, 16])
3 torch.Size([2, 2048, 8, 8])


In [29]:
class XceptionBranch(nn.Module):
  def __init__(self, model_name="legacy_xception",pretrained=True):
    super().__init__()

    self.backbone = timm.create_model(
        model_name,
        pretrained=pretrained,
        features_only=True,
        out_indices=(1,2,3,4),
    )

  def forward(self, x):
      feats = self.backbone(x)

      return {
          "low":feats[0],
          "entry":feats[1],
          "middle":feats[2],
          "exit":feats[3],
      }

In [30]:
branch = XceptionBranch()

x = torch.randn(2,3,256,256)

with torch.no_grad():
  out = branch(x)

for k,v in out.items():
  print(k,v.shape)

low torch.Size([2, 128, 63, 63])
entry torch.Size([2, 256, 32, 32])
middle torch.Size([2, 728, 16, 16])
exit torch.Size([2, 2048, 8, 8])


In [31]:
class CrossAttention2D(nn.Module):
  def __init__(self, in_q, in_kv, dim=256):
    super().__init__()

    self.q_proj = nn.Conv2d(in_q, dim, kernel_size=1)
    self.k_proj = nn.Conv2d(in_kv, dim, kernel_size=1)
    self.v_proj = nn.Conv2d(in_kv, dim, kernel_size=1)

    self.out_proj = nn.Conv2d(dim, dim, kernel_size=1)

  def forward(self, q_feat, kv_feat):
    B, _, H, W = q_feat.shape

    if kv_feat.shape[-2:] != (H,W):
      kv_feat = F.interpolate(
          kv_feat,
          size=(H,W),
          mode="bilinear",
          align_corners=False,
      )

    q = self.q_proj(q_feat)
    k = self.k_proj(kv_feat)
    v = self.v_proj(kv_feat)

    q = q.flatten(2).transpose(1, 2)
    k = k.flatten(2)
    v = v.flatten(2).transpose(1,2)

    attn = torch.bmm(q,k) / (q.shape[-1]**0.5)
    attn = torch.softmax(attn, dim=-1)

    out = torch.bmm(attn, v)

    out = out.transpose(1,2).reshape(B, -1, H, W)
    out = self.out_proj(out)

    return out

In [32]:
attn = CrossAttention2D(in_q=256, in_kv=256, dim = 256)
tex = torch.randn(2,256,32,32)
dl = torch.randn(2, 256, 32, 32)

out = attn(tex, dl)
out.shape

torch.Size([2, 256, 32, 32])

In [33]:
from IPython.core import oinspect
class ResidualCrossAttention2D(nn.Module):
  def __init__(self, in_q, in_kv, dim=256, add_kv_residual=True):
    super().__init__()

    self.attn = CrossAttention2D(in_q=in_q, in_kv=in_kv, dim=dim,)
    self.q_res = nn.Conv2d(in_q, dim, kernel_size=1)
    self.kv_res = nn.Conv2d(in_kv, dim, kernel_size=1)
    self.add_kv_residual=add_kv_residual

    self.norm = nn.BatchNorm2d(dim)
    self.act = nn.ReLU(inplace = True)

  def forward(self, q_feat, kv_feat):
    B, _, H, W = q_feat.shape

    if kv_feat.shape[-2:] != (H, W):
      kv_feat_resized = F.interpolate(
          kv_feat,
          size=(H,W),
          mode="bilinear",
          align_corners=False,
      )
    else:
      kv_feat_resized = kv_feat

    out = self.attn(q_feat, kv_feat)
    out = out + self.q_res(q_feat)

    if self.add_kv_residual:
      out = out + self.kv_res(kv_feat_resized)

    out = self.norm(out)
    out = self.act(out)

    return out

In [34]:
class SRINet(nn.Module):
  def __init__(self, model_name="legacy_xception",pretrained=True,attn_dim=256,num_classes=2):
    super().__init__()

    self.image_branch = XceptionBranch(model_name=model_name, pretrained=pretrained,)
    self.texture_branch = XceptionBranch(model_name=model_name, pretrained=pretrained,)
    self.direct_branch = XceptionBranch(model_name=model_name, pretrained=pretrained,)
    self.specular_branch = XceptionBranch(model_name=model_name, pretrained=pretrained,)

    self.cross_td = ResidualCrossAttention2D(in_q=256,in_kv=256, dim=attn_dim, add_kv_residual=True,)

    self.td_middle = nn.Sequential(
        nn.Conv2d(attn_dim, 728, kernel_size=1, bias=False),
        nn.BatchNorm2d(728),
        nn.ReLU(inplace=True),
    )

    self.cross_std = ResidualCrossAttention2D(in_q=728,in_kv=728, dim=attn_dim, add_kv_residual=True,)

    self.std_exit = nn.Sequential(
        nn.Conv2d(attn_dim, 2048, kernel_size=1, bias = False),
        nn.BatchNorm2d(2048),
        nn.ReLU(inplace=True),
    )

    self.pool = nn.AdaptiveAvgPool2d(1)

    self.classifier = nn.Sequential(
        nn.Linear(2048+2048,512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )

  def forward(self, image, texture_uv, direct_uv, specular_uv):
    img_feat = self.image_branch(image)
    tex_feat = self.texture_branch(texture_uv)
    dir_feat = self.direct_branch(direct_uv)
    spr_feat = self.specular_branch(specular_uv)

    f_td = self.cross_td(
        tex_feat["entry"],
        dir_feat["entry"],
    )

    f_td_mid = self.td_middle(f_td)

    if f_td_mid.shape[-2:] != spr_feat["middle"].shape[-2]:
      f_td_mid = F.interpolate(
          f_td_mid,
          size = spr_feat["middle"].shape[-2:],
          mode = "bilinear",
          align_corners=False,
        )

    f_std = self.cross_std(f_td_mid, spr_feat["middle"],)

    f_std_exit = self.std_exit(f_std)

    if f_std_exit.shape[-2:] != img_feat["exit"].shape[-2]:
      f_std_exit = F.interpolate(
          f_std_exit,
          size = img_feat["exit"].shape[-2:],
          mode = "bilinear",
          align_corners=False,
        )

    img_vec = self.pool(img_feat["exit"]).flatten(1)
    std_vec = self.pool(f_std_exit).flatten(1)

    fused = torch.cat([img_vec, std_vec],dim=1)

    logits = self.classifier(fused)

    return logits

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [36]:
model = SRINet(
    model_name = "legacy_xception",
    pretrained = True,
    attn_dim = 256,
    num_classes = 2,
).to(device)
model.eval()

SRINet(
  (image_branch): XceptionBranch(
    (backbone): FeatureHookNet(
      (body): Xception(
        (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act1): ReLU(inplace=True)
        (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
        (block1): Block(
          (skip): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (skipbn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (rep): Sequential(
            (0): SeparableConv2d(
              (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
              (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
            )

In [37]:
B = 2
H = 256
W = 256
image = torch.randn(B, 3 , H, W).to(device)
texture_uv = torch.randn(B, 3, H, W).to(device)
direct_uv = torch.randn(B, 3, H, W).to(device)
specular_uv =  torch.randn(B, 3, H, W).to(device)

with torch.no_grad():
  logits = model(
      image=image,
      texture_uv=texture_uv,
      direct_uv=direct_uv,
      specular_uv=specular_uv,
  )
print(logits.shape)
print(logits)


torch.Size([2, 2])
tensor([[-0.0157,  0.0179],
        [-0.0321,  0.0211]], device='cuda:0')
